In [ ]:
## Import necessary modules
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn import metrics
#from sklearn.metrics import plot_confusion_matrix
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.model_selection import RepeatedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve, matthews_corrcoef
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sn
import seaborn as sns
import pandas as pd
import csv
import math
from sklearn.feature_selection import SelectKBest, chi2
from sklearn import preprocessing
from sklearn.model_selection import cross_val_score,KFold,cross_val_predict
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score
import sklearn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/iSCuus_paper/Test DS/isucc-Test_selected_SFLA.csv')

In [ ]:
# ============================================================
# Import necessary modules
# ============================================================

import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# ============================================================
# Handle missing and infinite values
# ============================================================

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(999, inplace=True)


# ============================================================
# Split features and labels
# ============================================================

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

print(f"Original data shape: {X.shape}")
print(f"Original class distribution: {Counter(y)}")


# ============================================================
# Train-test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# Feature scaling
# Fit ONLY on training data
# Then transform training and testing data
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


print(f"\nFinal training shape: {X_train.shape}")
print(f"Final testing shape: {X_test.shape}")

TCN-GLU Model

In [ ]:
import numpy as np
import tensorflow as tf
from keras import layers, regularizers
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam


# ============================================================
# SwiGLU Layer
# ============================================================
class DenseSwiGLU(layers.Layer):

    def __init__(self, units, beta=10.0, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.beta = beta

        self.gate = layers.Dense(
            units,
            activation=None
        )

        self.value = layers.Dense(
            units,
            activation=None
        )

    def call(self, x):

        g = self.gate(x)

        # Swish activation
        gate_swish = g * tf.nn.sigmoid(self.beta * g)

        v = self.value(x)

        # Gated output
        return gate_swish * v


# ============================================================
# TCN + SwiGLU Model
# ============================================================
def build_tcn_swiglu_model(
        input_dim,
        l1=1e-3,
        dropout=0.15,
        beta=10.0):

    inputs = tf.keras.Input(
        shape=(input_dim, 1)
    )

    # ========================================================
    # TCN BLOCK 1 - dilation = 1
    # ========================================================
    x = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        dilation_rate=1,
        kernel_regularizer=regularizers.l1(l1)
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # ========================================================
    # TCN BLOCK 2 - dilation = 2
    # ========================================================
    x = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        dilation_rate=2,
        kernel_regularizer=regularizers.l1(l1)
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # ========================================================
    # TCN BLOCK 3 - dilation = 4
    # ========================================================
    x = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        dilation_rate=4,
        kernel_regularizer=regularizers.l1(l1)
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    """
    # ========================================================
    # TCN BLOCK 4 - dilation = 8
    # ========================================================
    x = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        dilation_rate=8,
        kernel_regularizer=regularizers.l1(l1)
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    """
    # ========================================================
    # Dropout
    # ========================================================
    x = layers.Dropout(dropout)(x)

    # ========================================================
    # Flatten before SwiGLU
    # ========================================================
    x = layers.Flatten()(x)

    # ========================================================
    # SwiGLU
    # ========================================================
    glu = DenseSwiGLU(
        units=32,
        beta=beta
    )(x)

    # Concatenate original features + gated features
    x = layers.Concatenate()([x, glu])

    # ========================================================
    # Fully Connected Layer
    # ========================================================
    x = layers.Dense(32)(x)

    x = layers.Activation("relu")(x)

    x = layers.Dropout(dropout)(x)

    # ========================================================
    # Output
    # ========================================================
    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    return tf.keras.Model(
        inputs,
        outputs
    )


# ============================================================
# Build Model
# ============================================================
model = build_tcn_swiglu_model(
    input_dim=X_train.shape[1],
    beta=10.0
)


# ============================================================
# Compile
# ============================================================
learning_rate = 0.0001

optimizer = Adam(
    learning_rate=learning_rate
)

model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# Metric Lists
# ============================================================
accuracy_list = []
f1_list = []
precision_list = []
recall_list = []
sensitivity_list = []
specificity_list = []
mcc_list = []
auc_list = []


# ============================================================
# 10-Fold Stratified Cross Validation
# ============================================================
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=2
)


for train_index, val_index in skf.split(
        X_train,
        y_train):

    X_train_fold = X_train[train_index]
    X_val_fold = X_train[val_index]

    y_train_fold = y_train[train_index]
    y_val_fold = y_train[val_index]

    # --------------------------------------------------------
    # Callbacks
    # --------------------------------------------------------
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5
    )

    callbacks = [
        early_stop,
        reduce_lr
    ]

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------
    history = model.fit(
        X_train_fold,
        y_train_fold,
        epochs=60,
        batch_size=64,
        verbose=0,
        validation_data=(
            X_val_fold,
            y_val_fold
        ),
        callbacks=callbacks
    )

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------
    y_val_pred = model.predict(
        X_val_fold,
        verbose=0
    ).ravel()

    y_val_pred_binary = (
        y_val_pred > 0.5
    ).astype(int)

    # --------------------------------------------------------
    # Confusion Matrix
    # --------------------------------------------------------
    cm = confusion_matrix(
        y_val_fold,
        y_val_pred_binary
    )

    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]
    TP = cm[1, 1]

    # --------------------------------------------------------
    # AUC
    # --------------------------------------------------------
    auc = roc_auc_score(
        y_val_fold,
        y_val_pred
    )

    auc_list.append(auc)

    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------
    accuracy = (
        (TP + TN) /
        float(TP + TN + FP + FN)
    )

    accuracy_list.append(accuracy)

    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------
    f1 = (
        2 * TP /
        float(2 * TP + FP + FN)
        if (2 * TP + FP + FN) != 0
        else 0.0
    )

    f1_list.append(f1)

    # --------------------------------------------------------
    # Precision
    # --------------------------------------------------------
    precision = (
        TP / float(TP + FP)
        if (TP + FP) != 0
        else 0.0
    )

    precision_list.append(precision)

    # --------------------------------------------------------
    # Recall
    # --------------------------------------------------------
    recall = (
        TP / float(TP + FN)
        if (TP + FN) != 0
        else 0.0
    )

    recall_list.append(recall)

    # --------------------------------------------------------
    # Sensitivity
    # --------------------------------------------------------
    sensitivity = (
        TP / float(TP + FN)
        if (TP + FN) != 0
        else 0.0
    )

    sensitivity_list.append(sensitivity)

    # --------------------------------------------------------
    # Specificity
    # --------------------------------------------------------
    specificity = (
        TN / float(TN + FP)
        if (TN + FP) != 0
        else 0.0
    )

    specificity_list.append(specificity)

    # --------------------------------------------------------
    # MCC
    # --------------------------------------------------------
    denominator = np.sqrt(
        (TP + FP) *
        (TP + FN) *
        (TN + FP) *
        (TN + FN)
    )

    mcc = (
        ((TP * TN) - (FP * FN)) /
        denominator
        if denominator != 0
        else 0.0
    )

    mcc_list.append(mcc)


# ============================================================
# Average Results
# ============================================================
avg_accuracy = np.mean(accuracy_list)
avg_f1 = np.mean(f1_list)
avg_precision = np.mean(precision_list)
avg_recall = np.mean(recall_list)
avg_sensitivity = np.mean(sensitivity_list)
avg_specificity = np.mean(specificity_list)
avg_mcc = np.mean(mcc_list)
avg_auc = np.mean(auc_list)


# ============================================================
# Print Results
# ============================================================
print("==========================================")
print("TCN-SwiGLU K-Fold Cross-Validation")
print("==========================================")

print("Accuracy    =", avg_accuracy)
print("F1 Score    =", avg_f1)
print("Precision   =", avg_precision)
print("Recall      =", avg_recall)
print("Sensitivity =", avg_sensitivity)
print("Specificity =", avg_specificity)
print("MCC         =", avg_mcc)
print("AUC         =", avg_auc)